In [2]:
!pip install pytorch-forecasting pytorch-lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 101.5 MB/s eta 0:00:00


imports

In [3]:
import os
import warnings
import pandas as pd
import numpy as np
import torch

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import NaNLabelEncoder, GroupNormalizer
from pytorch_forecasting.models import TemporalFusionTransformer
from pytorch_forecasting.metrics import MAE

import lightning.pytorch as pl
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint

import wandb
from lightning.pytorch.loggers import WandbLogger

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names",
    category=UserWarning,
)

wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mkapa22 (mkapa22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

configure kaggle credentials

In [4]:
import os
import json
from getpass import getpass

KAGGLE_USERNAME = input("Kaggle username: ").strip()
KAGGLE_API_KEY = getpass("Kaggle API key: ").strip()

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

kaggle_json_path = os.path.join(kaggle_dir, 'kaggle.json')
kaggle_config = {"username": KAGGLE_USERNAME, "key": KAGGLE_API_KEY}

with open(kaggle_json_path, 'w') as f:
    json.dump(kaggle_config, f)

os.chmod(kaggle_json_path, 0o600)

!pip install -q kagglehub
import kagglehub
kagglehub.login()


Kaggle username: mariamkapanadze
Kaggle API key: ··········


Kaggle credentials set.
Kaggle credentials successfully validated.


download competition

In [5]:
path = kagglehub.competition_download(
    "walmart-recruiting-store-sales-forecasting"
)

print(path)

100%|██████████| 2.70M/2.70M [00:00<00:00, 4.24MB/s]

Extracting files...
/root/.cache/kagglehub/competitions/walmart-recruiting-store-sales-forecasting


load csv files

In [6]:
import zipfile
import os

for file in os.listdir(path):
    if file.endswith(".zip"):
        file_path = os.path.join(path, file)

        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            zip_ref.extractall(path)

print(os.listdir(path))

['test.csv.zip', 'train.csv', 'train.csv.zip', 'sampleSubmission.csv', 'test.csv', 'stores.csv', 'sampleSubmission.csv.zip', 'features.csv.zip', 'features.csv']


prepare dataframe

In [7]:
import pandas as pd

train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
features = pd.read_csv(os.path.join(path, "features.csv"))
stores = pd.read_csv(os.path.join(path, "stores.csv"))
submission = pd.read_csv(os.path.join(path, "sampleSubmission.csv"))

In [8]:
train["Date"] = pd.to_datetime(train["Date"])
features["Date"] = pd.to_datetime(features["Date"])

train_model = train.merge(
    features,
    on=["Store","Date","IsHoliday"],
    how="left"
)

train_model = train_model.merge(
    stores,
    on="Store",
    how="left"
)

feature engineering

In [9]:
train_model["year"] = train_model["Date"].dt.year
train_model["month"] = train_model["Date"].dt.month
train_model["week"] = train_model["Date"].dt.isocalendar().week.astype(int)
train_model["dayofweek"] = train_model["Date"].dt.dayofweek
train_model["is_weekend"] = (train_model["dayofweek"] >= 5).astype(int)

markdown_cols = [
    "MarkDown1",
    "MarkDown2",
    "MarkDown3",
    "MarkDown4",
    "MarkDown5",
]

train_model[markdown_cols] = train_model[markdown_cols].fillna(0)

train_model["weight"] = train_model["IsHoliday"].astype(float) * 4 + 1


In [10]:
train_model.head()

train_model.shape

train_model.columns

Index(['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Temperature',
       'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4',
       'MarkDown5', 'CPI', 'Unemployment', 'Type', 'Size', 'year', 'month',
       'week', 'dayofweek', 'is_weekend', 'weight'],
      dtype='object')

In [11]:
train_model = train_model.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

train_model["time_idx"] = (
    train_model["Date"] - train_model["Date"].min()
).dt.days // 7

In [12]:
categorical_cols = ["Store", "Dept", "Type", "IsHoliday"]

for col in categorical_cols:
    train_model[col] = train_model[col].astype(str)

## Mount Google Drive for Checkpoints

In [13]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


validation split

In [14]:
max_time = train_model["time_idx"].max()

training_cutoff = max_time - 31

train_df = train_model[
    train_model.time_idx <= training_cutoff
].copy()

val_df = train_model[
    train_model.time_idx > training_cutoff
].copy()

print(train_df["Date"].min(), train_df["Date"].max())
print(val_df["Date"].min(), val_df["Date"].max())

2010-02-05 00:00:00 2012-03-23 00:00:00
2012-03-30 00:00:00 2012-10-26 00:00:00


In [15]:
max_encoder_length = 26
max_prediction_length = 31

**Note:** `GroupNormalizer` is not used as `target_normalizer` here -- the
'
    'dataset below relies on `TimeSeriesDataSet`'s default `EncoderNormalizer` for
'
    'the target instead. This cell used to build a `GroupNormalizer` and throw it
'
    'away without assigning it to anything, which did nothing.

build timeseries dataset

In [16]:
train_df = train_df.dropna(
    subset=["Weekly_Sales"]
).copy()

In [18]:
val_df = val_df.dropna(
    subset=["Weekly_Sales"]
).copy()

In [19]:
print("train_df shape:", train_df.shape)

print(
    "Weekly_Sales NaN:",
    train_df["Weekly_Sales"].isna().sum()
)

print(
    "Weekly_Sales infinite:",
    np.isinf(train_df["Weekly_Sales"]).sum()
)

print(
    train_df[
        train_df["Weekly_Sales"].isna()
    ].head()
)

train_df shape: (329817, 23)
Weekly_Sales NaN: 0
Weekly_Sales infinite: 0
Empty DataFrame
Columns: [Store, Dept, Date, Weekly_Sales, IsHoliday, Temperature, Fuel_Price, MarkDown1, MarkDown2, MarkDown3, MarkDown4, MarkDown5, CPI, Unemployment, Type, Size, year, month, week, dayofweek, is_weekend, weight, time_idx]
Index: []

[0 rows x 23 columns]


In [20]:
training = TimeSeriesDataSet(
    train_df,

    time_idx="time_idx",

    target="Weekly_Sales",

    group_ids=[
        "Store",
        "Dept"
    ],

    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,


    static_categoricals=[
        "Store",
        "Dept",
        "Type"
    ],


    static_reals=["Size"],


    time_varying_known_categoricals=[
        "IsHoliday"
    ],


    time_varying_known_reals=[
        "time_idx",
        "year",
        "month",
        "week",
        "dayofweek",
        "is_weekend"
    ],


    time_varying_unknown_reals=[
        "Weekly_Sales",
        "Temperature",
        "Fuel_Price",
        "CPI",
        "Unemployment",
        "MarkDown1",
        "MarkDown2",
        "MarkDown3",
        "MarkDown4",
        "MarkDown5"
    ],

    weight="weight",

    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,

    allow_missing_timesteps=True
)


/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 252 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__Store': '1', '__group_id__Dept': '51'}, {'__group_id__Store': '1', '__group_id__Dept': '77'}, {'__group_id__Store': '1', '__group_id__Dept': '78'}, {'__group_id__Store': '1', '__group_id__Dept': '99'}, {'__group_id__Store': '10', '__group_id__Dept': '51'}, {'__group_id__Store': '10', '__group_id__Dept': '77'}, {'__group_id__Store': '10', '__group_id__Dept': '78'}, {'__group_id__Store': '11', '__group_id__Dept': '48'}, {'__group_id__Store': '11', '__group_id__Dept': '77'}, {'__group_id__Store': '11', '__group_id__Dept': '78'}]
  warnings.warn(


In [21]:
validation = TimeSeriesDataSet.from_dataset(
    training,
    train_model,
    min_prediction_idx=training_cutoff + 1,
    predict=True,
    stop_randomization=True
)

/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 364 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__Store': '1', '__group_id__Dept': '47'}, {'__group_id__Store': '1', '__group_id__Dept': '77'}, {'__group_id__Store': '1', '__group_id__Dept': '99'}, {'__group_id__Store': '10', '__group_id__Dept': '45'}, {'__group_id__Store': '10', '__group_id__Dept': '47'}, {'__group_id__Store': '10', '__group_id__Dept': '77'}, {'__group_id__Store': '10', '__group_id__Dept': '78'}, {'__group_id__Store': '11', '__group_id__Dept': '45'}, {'__group_id__Store': '11', '__group_id__Dept': '47'}, {'__group_id__Store': '11', '__group_id__Dept': '48'}]
  warnings.warn(


In [22]:
train_loader = training.to_dataloader(
    train=True,
    batch_size=1024,
    num_workers=0,
    pin_memory=True
)

val_loader = validation.to_dataloader(
    train=False,
    batch_size=1024,
    num_workers=0,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train batches: 156
Validation batches: 3


In [23]:
results = []


In [24]:
def run_tft_experiment(
    run_name,
    hidden_size,
    attention_heads,
    dropout,
    learning_rate,
    checkpoint_path=None  # path to a .ckpt to resume from, optional
):

    print("\n=====================")
    print(run_name)
    print("=====================")

    wandb.init(
        project="walmart-sales-forecasting",
        name=run_name,
        reinit=True,
        config={
            "model": "Temporal Fusion Transformer",
            "hidden_size": hidden_size,
            "attention_heads": attention_heads,
            "dropout": dropout,
            "learning_rate": learning_rate,
            "encoder_length": max_encoder_length,
            "prediction_length": max_prediction_length,
            "validation_timestamp": 31
        }
    )

    wandb_logger = WandbLogger(
        project="walmart-sales-forecasting",
        name=run_name
    )

    tft = TemporalFusionTransformer.from_dataset(
        training,
        learning_rate=learning_rate,
        hidden_size=hidden_size,
        attention_head_size=attention_heads,
        hidden_continuous_size=16,
        dropout=dropout,
        loss=MAE(),
        reduce_on_plateau_patience=5,
        # Layer normalization is already built into TFT: every Gated Residual
        # Network / GateAddNorm block in the architecture applies LayerNorm as
        # part of its "AddNorm" step (this is standard in the original TFT
        # paper). There is no separate `normalization_layer` constructor
        # argument in pytorch-forecasting -- that's why passing it raised a
        # TypeError before. Nothing else needs to be added to get layer
        # normalization; it's on for every run by default.
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="min"
    )

    checkpoint_dir = os.path.join(
        '/content/drive/MyDrive/Colab Notebooks', 'tft_checkpoints', run_name
    )
    checkpoint_callback = ModelCheckpoint(
        dirpath=checkpoint_dir,
        filename="{epoch}-{val_loss:.2f}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
        verbose=True
    )

    lr_monitor = LearningRateMonitor(logging_interval="epoch")

    trainer = Trainer(
        max_epochs=30,
        accelerator="auto",
        gradient_clip_val=0.1,
        callbacks=[early_stop, checkpoint_callback, lr_monitor],
        logger=wandb_logger,
        enable_progress_bar=True
    )

    trainer.fit(
        tft,
        train_loader,
        val_loader,
        ckpt_path=checkpoint_path
    )

    # Because the dataset was built with weight="weight" (holiday weeks x5),
    # this val_loss is a weighted MAE -- i.e. the same WMAE metric Kaggle
    # scores this competition with. That's the number to compare against
    # ~1500 as your target, and against the historical leaderboard: the
    # original competition winner scored ~2301 WMAE on the private
    # leaderboard, with most solid public solutions landing in the
    # 2100-2900 range. A validation WMAE near 1500 would be excellent --
    # ambitious but worth aiming for. Just keep in mind this is measured on
    # your internal validation split, not an actual Kaggle submission, so
    # treat it as a strong proxy rather than a guaranteed leaderboard score.
    best_val_loss = None
    if checkpoint_callback.best_model_score is not None:
        best_val_loss = checkpoint_callback.best_model_score.item()

    result = {
        "run_name": run_name,
        "hidden_size": hidden_size,
        "attention_heads": attention_heads,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "MAE": best_val_loss,
        "best_checkpoint": checkpoint_callback.best_model_path,
    }
    results.append(result)

    wandb.log({"best_val_loss": best_val_loss})
    wandb.finish()

    return result


 GPU check

In [25]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "WARNING: no GPU detected. Trainer(accelerator=\"auto\") will silently "
        "fall back to CPU instead of erroring, which makes training 20-50x "
        "slower (minutes per batch of epochs instead of seconds). Go to "
        "Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4), "
        "save, reconnect, and re-run the cells above before continuing."
    )
    raise RuntimeError("No GPU available -- stopping before the training sweep runs on CPU.")


CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [26]:
import os
from getpass import getpass

os.environ["WANDB_API_KEY"] = getpass("W&B API key: ").strip()

W&B API key: ··········


In [27]:
import time

t0 = time.time()
batch = next(iter(train_loader))
print(f"First batch fetched in {time.time() - t0:.1f}s", flush=True)

First batch fetched in 0.5s


In [28]:
import time

print(">>> starting sanity test", flush=True)
t0 = time.time()

wandb.init(
    project="walmart-sales-forecasting",
    name="sanity_test",
    reinit=True,
    settings=wandb.Settings(console="off")  # stop wandb from capturing stdout, which can hide Lightning's progress bar
)
print(f">>> wandb.init done in {time.time()-t0:.1f}s", flush=True)

wandb_logger = WandbLogger(project="walmart-sales-forecasting", name="sanity_test")

tft_test = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.01,
    hidden_size=16,
    attention_head_size=1,
    hidden_continuous_size=16,
    dropout=0.1,
    loss=MAE(),
)
print(">>> model built", flush=True)

trainer_test = Trainer(
    max_epochs=1,
    limit_train_batches=5,   # only 5 batches, should take seconds
    limit_val_batches=2,
    accelerator="auto",
    logger=wandb_logger,
    enable_progress_bar=True,
    num_sanity_val_steps=0,  # skip Lightning's pre-flight validation check
)
print(">>> trainer built, calling fit()", flush=True)

trainer_test.fit(tft_test, train_loader, val_loader)
print(">>> fit() returned", flush=True)

wandb.finish()
print(f">>> sanity test done in {time.time()-t0:.1f}s total", flush=True)

>>> starting sanity test


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


>>> wandb.init done in 1.4s
>>> model built


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

>>> trainer built, calling fit()


...

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=47` in the `DataLoader` to improve performance.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py:321: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` 

>>> fit() returned


epoch,▁▁
train_loss_epoch,▁
trainer/global_step,▁▁
val_MAE,▁
val_MAPE,▁
val_RMSE,▁
val_SMAPE,▁
val_loss,▁
epoch,0
train_loss_epoch,5218.0874
trainer/global_step,4


>>> sanity test done in 8.4s total


In [29]:
import glob
import re

def reconstruct_result(run_name, hidden_size, attention_heads, dropout, learning_rate):
    ckpt_dir = f'/content/drive/MyDrive/Colab Notebooks/tft_checkpoints/{run_name}'
    ckpts = glob.glob(os.path.join(ckpt_dir, '*.ckpt'))
    if not ckpts:
        return None
    best_ckpt = ckpts[0]  # save_top_k=1, so there's only one
    match = re.search(r'val_loss=([\d.]+)\.ckpt', best_ckpt)
    val_loss = float(match.group(1)) if match else None
    return {
        "run_name": run_name, "hidden_size": hidden_size,
        "attention_heads": attention_heads, "dropout": dropout,
        "learning_rate": learning_rate, "MAE": val_loss,
        "best_checkpoint": best_ckpt,
    }

results = []
for cfg in run_configs[:2]:  # run1 and run2 already finished before the disconnect
    r = reconstruct_result(**cfg)
    if r:
        results.append(r)
print(results)

NameError: name 'run_configs' is not defined

 hyperparameter sweep runs

In [31]:
run_configs = [
    dict(run_name="TFT_run1_small",          hidden_size=16, attention_heads=1, dropout=0.10, learning_rate=0.01),
    dict(run_name="TFT_run2_medium",         hidden_size=32, attention_heads=2, dropout=0.10, learning_rate=0.01),
    dict(run_name="TFT_run3_medium_dropout", hidden_size=32, attention_heads=2, dropout=0.20, learning_rate=0.01),
    dict(run_name="TFT_run4_large",          hidden_size=64, attention_heads=4, dropout=0.10, learning_rate=0.01),
    dict(run_name="TFT_run5_large_lowlr",    hidden_size=64, attention_heads=4, dropout=0.10, learning_rate=0.003),
    dict(run_name="TFT_run6_wide_dropout",   hidden_size=64, attention_heads=4, dropout=0.30, learning_rate=0.01),
    dict(run_name="TFT_run7_lowlr",          hidden_size=32, attention_heads=2, dropout=0.10, learning_rate=0.001),
    dict(run_name="TFT_run8_manyheads",      hidden_size=32, attention_heads=8, dropout=0.15, learning_rate=0.005),
]

import glob
import re

def reconstruct_result(run_name, hidden_size, attention_heads, dropout, learning_rate):
    ckpt_dir = f'/content/drive/MyDrive/Colab Notebooks/tft_checkpoints/{run_name}'
    ckpts = glob.glob(os.path.join(ckpt_dir, '*.ckpt'))
    if not ckpts:
        return None
    best_ckpt = ckpts[0]  # save_top_k=1, so there's only one
    match = re.search(r'val_loss=([\d.]+)\.ckpt', best_ckpt)
    val_loss = float(match.group(1)) if match else None
    return {
        "run_name": run_name, "hidden_size": hidden_size,
        "attention_heads": attention_heads, "dropout": dropout,
        "learning_rate": learning_rate, "MAE": val_loss,
        "best_checkpoint": best_ckpt,
    }

results = []
for cfg in run_configs[:2]:  # run1 and run2 already finished before the disconnect
    r = reconstruct_result(**cfg)
    if r:
        results.append(r)
print(results)


[{'run_name': 'TFT_run1_small', 'hidden_size': 16, 'attention_heads': 1, 'dropout': 0.1, 'learning_rate': 0.01, 'MAE': 2388.37, 'best_checkpoint': '/content/drive/MyDrive/Colab Notebooks/tft_checkpoints/TFT_run1_small/epoch=3-val_loss=2388.37.ckpt'}, {'run_name': 'TFT_run2_medium', 'hidden_size': 32, 'attention_heads': 2, 'dropout': 0.1, 'learning_rate': 0.01, 'MAE': 2074.06, 'best_checkpoint': '/content/drive/MyDrive/Colab Notebooks/tft_checkpoints/TFT_run2_medium/epoch=18-val_loss=2074.06.ckpt'}]


In [38]:
import torch
_original_load = torch.load
torch.load = lambda *args, **kwargs: _original_load(*args, **{**kwargs, "weights_only": False})

In [39]:
run3_ckpt_dir = '/content/drive/MyDrive/Colab Notebooks/tft_checkpoints/TFT_run3_medium_dropout'
run3_ckpts = glob.glob(os.path.join(run3_ckpt_dir, '*.ckpt'))
latest_ckpt = max(run3_ckpts, key=os.path.getmtime)
print("Resuming from:", latest_ckpt)

run_tft_experiment(**run_configs[2], checkpoint_path=latest_ckpt)

Resuming from: /content/drive/MyDrive/Colab Notebooks/tft_checkpoints/TFT_run3_medium_dropout/epoch=16-val_loss=1964.62.ckpt

TFT_run3_medium_dropout


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: Restoring states from the checkpoint path at /content/drive/MyDrive/Colab Notebooks/tft_checkpoints/TFT_run3_medium_dropout/epoch=16-val_loss=1964.62.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the che

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ MAE                             │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │  2.1 K │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    672 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  8.5 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 37.7 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 14.3 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  2.1 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     64 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  5.3 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  3.2 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     33 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 115 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 115 K                                                                                                
Total estimated model params size (MB): 0.463                                                                      
Modules in train mode: 594                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

INFO: Restored all states from the checkpoint at /content/drive/MyDrive/Colab Notebooks/tft_checkpoints/TFT_run3_medium_dropout/epoch=16-val_loss=1964.62.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restored all states from the checkpoint at /content/drive/MyDrive/Colab Notebooks/tft_checkpoints/TFT_run3_medium_dropout/epoch=16-val_loss=1964.62.ckpt


...

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=47` in the `DataLoader` to improve performance.

INFO: Epoch 17, global step 2808: 'val_loss' was not in top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 17, global step 2808: 'val_loss' was not in top 1
INFO: Epoch 18, global step 2964: 'val_loss' was not in top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 18, global step 2964: 'val_loss' was not in top 1
INFO: Epoch 19, global step 3120: 'val_loss' was not in top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 19, global step 3120: 'val_loss' was not in top 1
INFO: Epoch 20, global step 3276: 'val_loss' was not in top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 20, global step 3276: 'val_loss' was not in top 1
INFO: Epoch 21, global step 3432: 'val_loss' reached 1938.08728 (best 1938.08728), saving model to '/content/drive/MyDrive/Colab Notebooks/tft_checkpoints/TFT_run3_medium_dropout/epoch=21-val_loss=1938.09.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 21, global step 3432: 'val_loss' reached 1938.08728 (best 1938.08728), saving model to 

best_val_loss,▁
epoch,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇███
lr-Adam,▁▁▁▁▁▁▁▁▁▁▁
train_loss_epoch,█▇▆▅▄▅▄▃▃▁▂
train_loss_step,▅▅▇▂▄▃▃▃▃▁▅▆█▁▂▃▃▂▂▁▂▃▃▄▂▂▅▁▂▃▄▁▃▂
trainer/global_step,▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇████
val_MAE,▅▅▆▄▂▁█▅▄▃█
val_MAPE,▁▆▆▂▅▁▅▇█▃▅
val_RMSE,▆▆█▆▃▁▇▇▅▂█
val_SMAPE,█▃▃▃▁▁▃▂▂▃▆
+1,...


{'run_name': 'TFT_run3_medium_dropout',
 'hidden_size': 32,
 'attention_heads': 2,
 'dropout': 0.2,
 'learning_rate': 0.01,
 'MAE': 1916.468017578125,
 'best_checkpoint': '/content/drive/MyDrive/Colab Notebooks/tft_checkpoints/TFT_run3_medium_dropout/epoch=22-val_loss=1916.47.ckpt'}

In [40]:
for cfg in run_configs[3:]:
    run_tft_experiment(**cfg)


TFT_run4_large


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ MAE                             │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │  2.1 K │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    672 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │ 13.8 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 59.6 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 23.4 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │ 33.3 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │ 33.3 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  8.3 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │    128 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │ 20.9 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │ 10.4 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  8.4 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  8.4 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     65 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 305 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 305 K                                                                                                
Total estimated model params size (MB): 1.223                                                                      
Modules in train mode: 598                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=47` in the `DataLoader` to improve performance.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=47` in the `DataLoader` to improve performance.

INFO: Epoch 0, global step 156: 'val_loss' reached 2807.24463 (best 2807.24463), saving model to '/content/drive/MyDrive/Colab Notebooks/tft_checkpoints/TFT_run8_manyheads/epoch=0-val_loss=2807.24.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 0, global step 156: 'val_loss' reached 2807.24463 (best 2807.24463), saving model to '/content/drive/MyDrive/Colab Notebooks/tft_checkpoints/TFT_run8_manyheads/epoch=0-val_loss=2807.24.ckpt' as top 1
INFO: Epoch 1, global step 312: 'val_loss' reached 2468.33521 (best 2468.33521), saving model to '/content/drive/MyDrive/Colab Notebooks/tft_checkpoints/TFT_run8_manyheads/epoch=1-val_loss=2468.34.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 1, global step 312: 'val_loss' reached 2468.33521 (best 2468.33521), saving model to '/content/drive/MyDrive/Colab Notebooks/tft_checkpoints/TFT_run8_manyheads/epoch=1-val_loss=2468.34.ckpt' as top 1
INFO: Epoch 2, global step 468: 'val_loss' reached 2319.73633 (best 2319.73633

best_val_loss,▁
epoch,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇██
lr-Adam,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss_epoch,█▅▄▃▃▂▂▂▂▂▂▁▁▁▁▁
train_loss_step,█▇▇▅▆▅▄▃▄▄▃▃▃▂▂▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▂▁▁▁▁
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇████
val_MAE,█▄▃▂▃▂▄▂▂▂▁▂▃▂▂▂
val_MAPE,▄▅▆▃▂▁█▁▇▂▄▃▆▅▁▁
val_RMSE,█▄▃▂▃▂▄▃▃▂▁▂▃▂▃▂
val_SMAPE,█▅▂▄▄▃▃▄▅▄▁▂▃▃▅▄
+1,...


List checkpoints saved in Google Drive for a given run:

In [ ]:
run_name_to_check = "TFT_run1_small"

checkpoints_dir = os.path.join('/content/drive/MyDrive/Colab Notebooks', 'tft_checkpoints', run_name_to_check)

if os.path.exists(checkpoints_dir):
    print(f"Checkpoints for {run_name_to_check} found at: {checkpoints_dir}")
    for root, dirs, files in os.walk(checkpoints_dir):
        for file in files:
            if file.endswith('.ckpt'):
                print(os.path.join(root, file))
else:
    print(f"No checkpoints found at {checkpoints_dir}.")


In [41]:
results_df = pd.DataFrame(results)

wandb.init(
    project="walmart-sales-forecasting",
    name="TFT_summary",
    reinit=True
)

wandb.log(
    {
        "best_TFT_MAE": results_df["MAE"].min()
    }
)

wandb.finish()

results_df.sort_values("MAE")


best_TFT_MAE,▁
best_TFT_MAE,1916.46802


,run_name,hidden_size,attention_heads,dropout,learning_rate,MAE,best_checkpoint
2,TFT_run3_medium_dropout,32,2,0.20,0.010,1916.468018,/content/drive/MyDrive/Colab Notebooks/tft_che...
5,TFT_run6_wide_dropout,64,4,0.30,0.010,2029.196655,/content/drive/MyDrive/Colab Notebooks/tft_che...
4,TFT_run5_large_lowlr,64,4,0.10,0.003,2032.109009,/content/drive/MyDrive/Colab Notebooks/tft_che...
1,TFT_run2_medium,32,2,0.10,0.010,2074.060000,/content/drive/MyDrive/Colab Notebooks/tft_che...
7,TFT_run8_manyheads,32,8,0.15,0.005,2148.316162,/content/drive/MyDrive/Colab Notebooks/tft_che...
6,TFT_run7_lowlr,32,2,0.10,0.001,2356.019287,/content/drive/MyDrive/Colab Notebooks/tft_che...
0,TFT_run1_small,16,1,0.10,0.010,2388.370000,/content/drive/MyDrive/Colab Notebooks/tft_che...
3,TFT_run4_large,64,4,0.10,0.010,2685.782227,/content/drive/MyDrive/Colab Notebooks/tft_che...


In [42]:
print(len(results))
for r in results:
    print(r["run_name"], r["MAE"])

8
TFT_run1_small 2388.37
TFT_run2_medium 2074.06
TFT_run3_medium_dropout 1916.468017578125
TFT_run4_large 2685.7822265625
TFT_run5_large_lowlr 2032.1090087890625
TFT_run6_wide_dropout 2029.1966552734375
TFT_run7_lowlr 2356.019287109375
TFT_run8_manyheads 2148.316162109375


pipeline

In [43]:
train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
features = pd.read_csv(os.path.join(path, "features.csv"))
stores = pd.read_csv(os.path.join(path, "stores.csv"))

In [44]:
train["Date"] = pd.to_datetime(train["Date"])
features["Date"] = pd.to_datetime(features["Date"])

train_model = train.merge(features, on=["Store","Date","IsHoliday"], how="left")
train_model = train_model.merge(stores, on="Store", how="left")

train_model["year"] = train_model["Date"].dt.year
train_model["month"] = train_model["Date"].dt.month
train_model["week"] = train_model["Date"].dt.isocalendar().week.astype(int)
train_model["dayofweek"] = train_model["Date"].dt.dayofweek
train_model["is_weekend"] = (train_model["dayofweek"] >= 5).astype(int)

markdown_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
train_model[markdown_cols] = train_model[markdown_cols].fillna(0)
train_model["weight"] = train_model["IsHoliday"].astype(float) * 4 + 1

train_model = train_model.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
train_model["time_idx"] = (train_model["Date"] - train_model["Date"].min()).dt.days // 7

for col in ["Store", "Dept", "Type", "IsHoliday"]:
    train_model[col] = train_model[col].astype(str)

In [45]:
max_time = train_model["time_idx"].max()
training_cutoff = max_time - 31

train_df = train_model[train_model.time_idx <= training_cutoff].copy()
train_df = train_df.dropna(subset=["Weekly_Sales"]).copy()

max_encoder_length = 26
max_prediction_length = 31

training = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target="Weekly_Sales",
    group_ids=["Store", "Dept"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=["Store", "Dept", "Type"],
    static_reals=["Size"],
    time_varying_known_categoricals=["IsHoliday"],
    time_varying_known_reals=["time_idx", "year", "month", "week", "dayofweek", "is_weekend"],
    time_varying_unknown_reals=["Weekly_Sales", "Temperature", "Fuel_Price", "CPI", "Unemployment",
                                  "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"],
    weight="weight",
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True
)

/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 252 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__Store': '1', '__group_id__Dept': '51'}, {'__group_id__Store': '1', '__group_id__Dept': '77'}, {'__group_id__Store': '1', '__group_id__Dept': '78'}, {'__group_id__Store': '1', '__group_id__Dept': '99'}, {'__group_id__Store': '10', '__group_id__Dept': '51'}, {'__group_id__Store': '10', '__group_id__Dept': '77'}, {'__group_id__Store': '10', '__group_id__Dept': '78'}, {'__group_id__Store': '11', '__group_id__Dept': '48'}, {'__group_id__Store': '11', '__group_id__Dept': '77'}, {'__group_id__Store': '11', '__group_id__Dept': '78'}]
  warnings.warn(


In [47]:
import os
import mlflow
from mlflow.tracking import MlflowClient

mlruns_dir = "/content/drive/MyDrive/Colab Notebooks/mlruns"
os.makedirs(mlruns_dir, exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:////{mlruns_dir.lstrip('/')}/mlflow.db")

experiment_name = "walmart-tft-sweep"
artifact_location = f"file:{mlruns_dir}/artifacts"

client = MlflowClient()
experiment = client.get_experiment_by_name(experiment_name)
if experiment is None:
    client.create_experiment(experiment_name, artifact_location=artifact_location)

mlflow.set_experiment(experiment_name)
print("Tracking URI:", mlflow.get_tracking_uri())

2026/07/12 15:21:31 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/12 15:21:31 INFO mlflow.store.db.utils: Updating database tables


Tracking URI: sqlite:////content/drive/MyDrive/Colab Notebooks/mlruns/mlflow.db


save the artifacts

In [50]:
import pickle

artifact_dir = "/content/mlflow_pipeline_artifacts"
os.makedirs(artifact_dir, exist_ok=True)

# last max_encoder_length weeks per (Store, Dept) -- gives the model history to encode
history_df = (
    train_model.sort_values(["Store", "Dept", "Date"])
    .groupby(["Store", "Dept"], group_keys=False)
    .tail(max_encoder_length)
)
history_df.to_parquet(os.path.join(artifact_dir, "history_df.parquet"))

# stores.csv / features.csv are needed to enrich raw test.csv the same way train.csv was enriched
stores.to_csv(os.path.join(artifact_dir, "stores.csv"), index=False)
features.to_csv(os.path.join(artifact_dir, "features.csv"), index=False)

# the exact date origin used for time_idx during training -- inference must match it
train_min_date = train_model["Date"].min()
with open(os.path.join(artifact_dir, "train_min_date.txt"), "w") as f:
    f.write(train_min_date.isoformat())

# the TimeSeriesDataSet itself, used as the schema template for from_dataset() at inference
with open(os.path.join(artifact_dir, "training_dataset.pkl"), "wb") as f:
    pickle.dump(training, f)

print("Pipeline artifacts saved to", artifact_dir)

Pipeline artifacts saved to /content/mlflow_pipeline_artifacts


define pipeline class

In [89]:
class WalmartTFTPipeline(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        import pandas as pd
        import pickle
        from pytorch_forecasting.models import TemporalFusionTransformer

        self.model = TemporalFusionTransformer.load_from_checkpoint(
            context.artifacts["checkpoint"]
        )
        self.model.eval()

        self.stores = pd.read_csv(context.artifacts["stores_csv"])
        self.features = pd.read_csv(context.artifacts["features_csv"])
        self.features["Date"] = pd.to_datetime(self.features["Date"])

        # CPI/Unemployment aren't populated by Kaggle for the later part of the
        # test period. Forward-fill (then back-fill as a safety net) per Store
        # using the full features timeline, BEFORE merging into test data --
        # these move slowly month to month so carrying the last known value
        # forward is standard practice for this dataset.
        self.features = self.features.sort_values(["Store", "Date"])
        for col in ["Temperature", "Fuel_Price", "CPI", "Unemployment"]:
            self.features[col] = self.features.groupby("Store")[col].transform(
                lambda x: x.ffill().bfill()
            )

        self.history_df = pd.read_parquet(context.artifacts["history_df"])

        with open(context.artifacts["training_dataset"], "rb") as f:
            self.training_dataset = pickle.load(f)

        with open(context.artifacts["train_min_date"]) as f:
            self.train_min_date = pd.Timestamp(f.read().strip())

    def _preprocess(self, raw_test_df):
        import pandas as pd

        df = raw_test_df.copy()
        df["Date"] = pd.to_datetime(df["Date"])

        df = df.merge(self.features, on=["Store", "Date", "IsHoliday"], how="left")
        df = df.merge(self.stores, on="Store", how="left")

        df["year"] = df["Date"].dt.year
        df["month"] = df["Date"].dt.month
        df["week"] = df["Date"].dt.isocalendar().week.astype(int)
        df["dayofweek"] = df["Date"].dt.dayofweek
        df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

        markdown_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
        df[markdown_cols] = df[markdown_cols].fillna(0)

        df["weight"] = df["IsHoliday"].astype(float) * 4 + 1
        df["Weekly_Sales"] = 0.0

        df["time_idx"] = (df["Date"] - self.train_min_date).dt.days // 7

        for col in ["Store", "Dept", "Type", "IsHoliday"]:
            df[col] = df[col].astype(str)

        return df

    def predict(self, context, model_input):
      import numpy as np
      import pandas as pd
      from pytorch_forecasting import TimeSeriesDataSet

      test_processed = self._preprocess(model_input)

      combined = pd.concat(
          [self.history_df, test_processed], ignore_index=True
      ).sort_values(["Store", "Dept", "Date"])

      # Anchor the decode window to the START of the test period (not the
      # series' end) so the encoder only ever sees real historical sales --
      # never the placeholder Weekly_Sales=0.0 rows from the test period
      # itself. Without this, pytorch-forecasting silently uses some of the
      # test period's fake zero-sales as fictional "recent history."
      test_start_time_idx = int(test_processed["time_idx"].min())

      pred_dataset = TimeSeriesDataSet.from_dataset(
          self.training_dataset,
          combined,
          predict=True,
          stop_randomization=True,
          min_prediction_idx=test_start_time_idx,
      )

      result = self.model.predict(
          pred_dataset, mode="prediction", return_index=True, batch_size=512
      )

      preds = result.output.numpy()
      index_df = result.index.copy()
      index_df["Store"] = index_df["Store"].astype(str)
      index_df["Dept"] = index_df["Dept"].astype(str)

      n_groups, horizon = preds.shape

      expanded = pd.DataFrame({
          "Store": np.repeat(index_df["Store"].values, horizon),
          "Dept": np.repeat(index_df["Dept"].values, horizon),
          "time_idx": np.repeat(index_df["time_idx"].values, horizon) + np.tile(np.arange(horizon), n_groups),
          "Weekly_Sales_Pred": preds.reshape(-1),
      })

      date_lookup = test_processed[["Store", "Dept", "time_idx", "Date"]].drop_duplicates()
      date_lookup["Store"] = date_lookup["Store"].astype(str)
      date_lookup["Dept"] = date_lookup["Dept"].astype(str)

      out = expanded.merge(date_lookup, on=["Store", "Dept", "time_idx"], how="inner")
      out["Id"] = out["Store"] + "_" + out["Dept"] + "_" + out["Date"].dt.strftime("%Y-%m-%d")

      return out[["Id", "Store", "Dept", "Date", "Weekly_Sales_Pred"]]

/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


log one pipeline per run

In [90]:
pip_requirements = [
    "pytorch-forecasting",
    "lightning",
    "torch",
    "pandas",
    "numpy",
]

logged_models = {}

for r in results:
    with mlflow.start_run(run_name=r["run_name"]) as run:
        mlflow.log_params({
            "hidden_size": r["hidden_size"],
            "attention_heads": r["attention_heads"],
            "dropout": r["dropout"],
            "learning_rate": r["learning_rate"],
        })
        mlflow.log_metric("MAE", r["MAE"])

        artifacts = {
            "checkpoint": r["best_checkpoint"],
            "history_df": os.path.join(artifact_dir, "history_df.parquet"),
            "stores_csv": os.path.join(artifact_dir, "stores.csv"),
            "features_csv": os.path.join(artifact_dir, "features.csv"),
            "training_dataset": os.path.join(artifact_dir, "training_dataset.pkl"),
            "train_min_date": os.path.join(artifact_dir, "train_min_date.txt"),
        }

        mlflow.pyfunc.log_model(
            artifact_path="pipeline",
            python_model=WalmartTFTPipeline(),
            artifacts=artifacts,
            pip_requirements=pip_requirements,
        )

        logged_models[r["run_name"]] = f"runs:/{run.info.run_id}/pipeline"
        print(f"Logged pipeline for {r['run_name']} (MAE={r['MAE']:.2f}), run_id={run.info.run_id}")

2026/07/12 16:05:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 16:05:47 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


2026/07/12 16:05:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 16:05:47 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


Logged pipeline for TFT_run1_small (MAE=2388.37), run_id=9b6a68a28f6f4f9fb39d6bcf5e44cc67


2026/07/12 16:05:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 16:05:48 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


Logged pipeline for TFT_run2_medium (MAE=2074.06), run_id=96c0980da7294f66944c521ac506e1ad


...

2026/07/12 16:05:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 16:05:49 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


Logged pipeline for TFT_run6_wide_dropout (MAE=2029.20), run_id=09c850a11967426ba46e38ebe1d721fa


2026/07/12 16:05:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 16:05:49 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


Logged pipeline for TFT_run7_lowlr (MAE=2356.02), run_id=033d32ed39e44c688df79548c9f85604


Logged pipeline for TFT_run8_manyheads (MAE=2148.32), run_id=dff44f0e186546ba8b3a0a12b5d30dc7


register only best one in the model registry

In [91]:
best_run = min(results, key=lambda r: r["MAE"])

with mlflow.start_run(run_name=best_run["run_name"] + "_fixed2") as run:
    mlflow.log_params({k: best_run[k] for k in ["hidden_size", "attention_heads", "dropout", "learning_rate"]})
    mlflow.log_metric("MAE", best_run["MAE"])

    artifacts = {
        "checkpoint": best_run["best_checkpoint"],
        "history_df": os.path.join(artifact_dir, "history_df.parquet"),
        "stores_csv": os.path.join(artifact_dir, "stores.csv"),
        "features_csv": os.path.join(artifact_dir, "features.csv"),
        "training_dataset": os.path.join(artifact_dir, "training_dataset.pkl"),
        "train_min_date": os.path.join(artifact_dir, "train_min_date.txt"),
    }

    mlflow.pyfunc.log_model(
        artifact_path="pipeline",
        python_model=WalmartTFTPipeline(),
        artifacts=artifacts,
        pip_requirements=["pytorch-forecasting", "lightning", "torch", "pandas", "numpy"],
    )
    fixed_model_uri = f"runs:/{run.info.run_id}/pipeline"

registered = mlflow.register_model(model_uri=fixed_model_uri, name="walmart-tft-best")
pipeline = mlflow.pyfunc.load_model(f"models:/walmart-tft-best/{registered.version}")
predictions = pipeline.predict(test)

model_pred_ids = set(predictions["Id"])
all_ids = set(submission["Id"])
print(f"Model predictions: {len(model_pred_ids)} rows ({len(model_pred_ids)/len(all_ids):.1%})")

2026/07/12 16:05:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 16:05:53 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


Registered model 'walmart-tft-best' already exists. Creating a new version of this model...
2026/07/12 16:05:54 WARNING mlflow.tracking._model_registry.fluent: Run with id 03b3aad8c9df47ea945efefae0b8e09d has no artifacts at artifact path 'pipeline', registering model based on models:/m-2b45aa85524b45d184c820afec9892b9 instead
Created version '8' of model 'walmart-tft-best'.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/timeseries/_times

Model predictions: 88925 rows (77.3%)


In [92]:
pipeline = mlflow.pyfunc.load_model(f"models:/walmart-tft-best/{registered.version}")
predictions = pipeline.predict(test)
predictions.head()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 306 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__Store': '1', '__group_id__Dept': '45'}, {'__group_id__Store': '1', '__group_id

,Id,Store,Dept,Date,Weekly_Sales_Pred
0,1_1_2012-12-28,1,1,2012-12-28,4663.938477
1,1_1_2013-01-04,1,1,2013-01-04,8000.968262
2,1_1_2013-01-11,1,1,2013-01-11,6528.283203
3,1_1_2013-01-18,1,1,2013-01-18,6857.753906
4,1_1_2013-01-25,1,1,2013-01-25,10899.207031


handling missing groups before submitting

In [93]:
train_model["week"] = train_model["Date"].dt.isocalendar().week.astype(int)

seasonal_fallback = (
    train_model.groupby(["Store", "Dept", "week"])["Weekly_Sales"]
    .mean()
    .reset_index()
    .rename(columns={"Weekly_Sales": "seasonal_fallback"})
)
seasonal_fallback["Store"] = seasonal_fallback["Store"].astype(str)
seasonal_fallback["Dept"] = seasonal_fallback["Dept"].astype(str)

overall_avg = train_model["Weekly_Sales"].mean()  # last-resort, for groups with zero history

build full submission

In [94]:
submission = submission.copy()  # original sampleSubmission.csv dataframe

submission[["Store", "Dept", "Date"]] = submission["Id"].str.split("_", expand=True)
submission["week"] = pd.to_datetime(submission["Date"]).dt.isocalendar().week.astype(int)

pred_lookup = predictions.set_index("Id")["Weekly_Sales_Pred"]
submission["Weekly_Sales"] = submission["Id"].map(pred_lookup)

# fill gaps for the dropped groups
missing_mask = submission["Weekly_Sales"].isna()
print(f"{missing_mask.sum()} rows need a fallback")

submission = submission.merge(seasonal_fallback, on=["Store", "Dept", "week"], how="left")
submission.loc[missing_mask, "Weekly_Sales"] = submission.loc[missing_mask, "seasonal_fallback"]
submission["Weekly_Sales"] = submission["Weekly_Sales"].fillna(overall_avg)

submission = submission[["Id", "Weekly_Sales"]]
submission.to_csv("/content/submission.csv", index=False)
submission.head()

26139 rows need a fallback


/tmp/ipykernel_1503/4223583643.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[37062.47  19119.465 19301.75  ...   619.34    537.97    653.63 ]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  submission.loc[missing_mask, "Weekly_Sales"] = submission.loc[missing_mask, "seasonal_fallback"]


,Id,Weekly_Sales
0,1_1_2012-11-02,37062.470
1,1_1_2012-11-09,19119.465
2,1_1_2012-11-16,19301.750
3,1_1_2012-11-23,19865.770
4,1_1_2012-11-30,23905.525


In [96]:
from google.colab import files
files.download("/content/submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [95]:
model_pred_ids = set(predictions["Id"])
all_ids = set(submission["Id"])
fallback_ids = all_ids - model_pred_ids

print(f"Model predictions: {len(model_pred_ids)} rows ({len(model_pred_ids)/len(all_ids):.1%})")
print(f"Fallback rows: {len(fallback_ids)} rows ({len(fallback_ids)/len(all_ids):.1%})")

Model predictions: 88925 rows (77.3%)
Fallback rows: 26139 rows (22.7%)


In [97]:
import numpy as np
import torch
from pytorch_forecasting.models import TemporalFusionTransformer
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.metrics import MAE

best_run = min(results, key=lambda r: r["MAE"])
model = TemporalFusionTransformer.load_from_checkpoint(best_run["best_checkpoint"])
model.eval()

# Use the SAME expand-and-merge logic as the pipeline's predict(), but on
# `validation` -- which has real, known Weekly_Sales -- so we can directly
# check whether the merge/index math is correct, independent of whether the
# model itself generalizes well into the future.
result = model.predict(validation, mode="prediction", return_index=True, batch_size=512)

preds = result.output.numpy()
index_df = result.index.copy()
n_groups, horizon = preds.shape

expanded = pd.DataFrame({
    "Store": np.repeat(index_df["Store"].astype(str).values, horizon),
    "Dept": np.repeat(index_df["Dept"].astype(str).values, horizon),
    "time_idx": np.repeat(index_df["time_idx"].values, horizon) + np.tile(np.arange(horizon), n_groups),
    "Weekly_Sales_Pred": preds.reshape(-1),
})

val_df_check = val_df.copy()
val_df_check["Store"] = val_df_check["Store"].astype(str)
val_df_check["Dept"] = val_df_check["Dept"].astype(str)

merged = expanded.merge(
    val_df_check[["Store", "Dept", "time_idx", "Weekly_Sales", "IsHoliday"]],
    on=["Store", "Dept", "time_idx"],
    how="inner"
)

print("Matched rows:", len(merged), "/ expected around", len(val_df))

weight = merged["IsHoliday"].astype(str).map({"True": 5, "False": 1}).fillna(1)
wmae = (weight * (merged["Weekly_Sales"] - merged["Weekly_Sales_Pred"]).abs()).sum() / weight.sum()
print("Reconstructed WMAE on validation set:", wmae)

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

Matched rows: 88734 / expected around 91753
Reconstructed WMAE on validation set: 1711.2753641795289


In [98]:
import pandas as pd
import numpy as np
from pytorch_forecasting.models import TemporalFusionTransformer
from pytorch_forecasting import TimeSeriesDataSet

model = TemporalFusionTransformer.load_from_checkpoint(best_run["best_checkpoint"])
model.eval()

# rebuild test_processed the same way the pipeline does, but inline so we can inspect it
test_processed = test.copy()
test_processed["Date"] = pd.to_datetime(test_processed["Date"])

features_ff = features.copy()
features_ff["Date"] = pd.to_datetime(features_ff["Date"])
features_ff = features_ff.sort_values(["Store", "Date"])
for col in ["Temperature", "Fuel_Price", "CPI", "Unemployment"]:
    features_ff[col] = features_ff.groupby("Store")[col].transform(lambda x: x.ffill().bfill())

test_processed = test_processed.merge(features_ff, on=["Store", "Date", "IsHoliday"], how="left")
test_processed = test_processed.merge(stores, on="Store", how="left")

print("NaNs after merge:\n", test_processed[["Temperature","Fuel_Price","CPI","Unemployment","Type","Size"]].isna().sum())
print("\nCPI range:", test_processed["CPI"].min(), "-", test_processed["CPI"].max())
print("Fuel_Price range:", test_processed["Fuel_Price"].min(), "-", test_processed["Fuel_Price"].max())

print("\nTraining CPI range:", train_model["CPI"].min(), "-", train_model["CPI"].max())
print("Training Fuel_Price range:", train_model["Fuel_Price"].min(), "-", train_model["Fuel_Price"].max())

NaNs after merge:
 Temperature     0
Fuel_Price      0
CPI             0
Unemployment    0
Type            0
Size            0
dtype: int64

CPI range: 131.2362258 - 228.9764563
Fuel_Price range: 2.872 - 4.125

Training CPI range: 126.064 - 227.2328068
Training Fuel_Price range: 2.472 - 4.468


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [99]:
sample_preds = predictions.copy()
sample_preds["Store"] = sample_preds["Store"].astype(str)
sample_preds["Dept"] = sample_preds["Dept"].astype(str)

sw_fallback = seasonal_fallback.copy()
sample_preds["week"] = pd.to_datetime(sample_preds["Date"]).dt.isocalendar().week.astype(int)
compare = sample_preds.merge(sw_fallback, on=["Store", "Dept", "week"], how="left")

print(compare[["Weekly_Sales_Pred", "seasonal_fallback"]].describe())
print("\nNegative predictions:", (compare["Weekly_Sales_Pred"] < 0).sum())
print("Predictions > 5x same-week-last-year:", (compare["Weekly_Sales_Pred"] > 5 * compare["seasonal_fallback"]).sum())
print("\nSample comparison:")
compare[["Id", "Weekly_Sales_Pred", "seasonal_fallback"]].sample(15, random_state=1)

       Weekly_Sales_Pred  seasonal_fallback
count       88925.000000       88744.000000
mean         9654.991211       16031.721182
std         15642.046875       21785.889928
min        -37668.203125        -449.000000
25%           910.702209        2370.945000
50%          3649.817383        7919.773333
75%         11650.624023       20420.970833
max        213194.390625      233447.293333

Negative predictions: 1650
Predictions > 5x same-week-last-year: 235

Sample comparison:


,Id,Weekly_Sales_Pred,seasonal_fallback
19983,18_40_2013-01-11,-1316.359375,58608.695000
58765,35_54_2013-05-10,2.899662,162.600000
28977,21_6_2013-01-18,423.103394,1600.500000
2768,10_28_2013-06-21,266.309692,887.666667
77588,45_34_2013-05-03,10088.521484,16083.646667
26462,20_41_2013-03-01,2903.603027,2312.653333
18696,17_83_2013-05-31,385.185974,558.016667
86800,8_9_2013-06-14,10203.785156,13773.683333
51749,31_91_2013-06-14,70048.015625,76600.790000
17928,17_41_2013-03-29,15.137383,-1.333333
